# Эмбеддинги и интерпретация CNN

Цель: исследовать признаки предобученной EfficientNet-B0, сравнить правильные и ошибочные предсказания и визуализировать области изображения, связанные с решением сети.

Вам даны RGB-изображения 224×224 и файл `predictions.csv`, в котором хранятся предсказания модели на валидационной части датасет ImageNet. Обучать модель, скачивать весь ImageNet или повторно классифицировать весь validation не требуется.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_distances
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

# Рабочая папка Jupyter — папка задания; внутри data находятся CSV и images/.
IMAGE_DIR = Path("data").resolve()
PREDICTIONS_PATH = IMAGE_DIR / "predictions.csv"
SEED = 42
BATCH_SIZE = 16
DEVICE = torch.device("cpu")
EXPECTED_IMAGE_SIZE = (224, 224)
WEIGHTS = EfficientNet_B0_Weights.IMAGENET1K_V1
CLASS_NAMES = WEIGHTS.meta["categories"]
model = efficientnet_b0(weights=WEIGHTS).to(DEVICE)
model.eval()
official_preprocess = WEIGHTS.transforms()
# Preprocessing из predictions.csv: без Resize/CenterCrop. Изображения уже предобработаны
preprocess = transforms.Compose([
    transforms.PILToTensor(),
    transforms.ConvertImageDtype(torch.float32),
    transforms.Normalize(mean=official_preprocess.mean, std=official_preprocess.std),
])
# размерность эмбеддинга
embedding_dim = model.classifier[1].in_features

predictions = pd.read_csv(PREDICTIONS_PATH)
predictions["correct"] = predictions["label"] == predictions["predicted_label"]
prediction_records = predictions.to_dict(orient="records")

def load_rgb(path):
    with Image.open(path) as source:
        if source.mode != "RGB" or source.size != EXPECTED_IMAGE_SIZE:
            raise ValueError(f"Ожидалось RGB {EXPECTED_IMAGE_SIZE}: {path}")
        return source.copy()

class PredictionImages(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return preprocess(load_rgb(IMAGE_DIR / self.records[index]["file_name"]))

print(f"Загружено {len(prediction_records)} записей; размер эмбеддинга: {embedding_dim}.")

## 1. Эмбеддинги правильных предсказаний и t-SNE

Эмбеддинг изображения — вектор чисел, описывающий признаки, которые нейросеть извлекла из изображения. Здесь используем вектор после глобального усреднения признаков и перед классификатором; это представление изображения, а не вероятности классов.

1. Для классов `[207, 281, 404, 569, 954]` выберите случайно, без повторений, по 34 **правильно предсказанных** изображения.
2. Реализуйте `extract_embeddings(records)`, который возвращает эмбеддинги для батча изображений. Для EfficientNet-B0 каждый такой вектор/эмбеддинг извлекается после `AdaptiveAvgPool2d` и перед блоком `classifier` (см. архитектуру `model`); сохраните порядок объектов.
3. Получите 5*34 = 170 эмбеддингов. Примените сначала PCA для уменьшения размерности, а затем t-SNE для 2D визуализации эмбеддингов. Подберите параметры количества компонент PCA и t-SNE perplexity, чтобы векторы картинок одного класса лежали близко друг к другу.

In [ ]:
VISUALIZED_CLASS_IDS = [207, 281, 404, 569, 954]
SAMPLES_PER_CLASS = 34
PCA_COMPONENTS = # YOUR CODE HERE
TSNE_PERPLEXITY = # YOUR CODE HERE

def extract_embeddings(records):
    """Возвращает numpy-массив размера (len(records), embedding_dim)."""
    pass
    # YOUR CODE HERE


# YOUR CODE HERE

#### Вопросы (в ноутбуке отвечать не нужно)

1. Чем эмбеддинг отличается от любого другого представления изображения в виде вектора? Например, если бы мы просто превратили двумерный массив чисел картинки в одномерный?
2. Зачем нужны и `model.eval()`, и отключение градиентов? Что изменится, если оставить режим обучения?
3. Для чего здесь PCA перед t-SNE? Какие ограничения есть у интерпретации расстояний и размеров кластеров t-SNE?

## 2. Ошибки и косинусные расстояния

Возьмите **все ошибочные** примеры тех же пяти истинных классов и извлеките эмбеддинги. Используйте правильные примеры задания 1, не выбирая их заново.

Для каждого истинного класса вычислите два средних косинусных расстояния:

- Между правильными примерами (только между разными изображениями).
- Между каждым ошибочным и каждым выбранным правильным примером этого класса.

Используйте исходные эмбеддинги, а не после PCA/t-SNE. Постройте таблицу: класс, число правильных и ошибок, два значения средних расстояний.
Сделайте вывод о величине разницы.

In [ ]:
# YOUR CODE HERE

#### Вопросы

1. Что такое косинусное расстояние, какой у нее диапазон значений
2. Как можно использовать косинусное расстояние в векторных БД?
3. Может ли изображение быть правильно классифицировано, хотя его эмбеддинг находится далеко от эмбеддингов других изображений того же класса? И наоборот: может ли близкий к ним эмбеддинг получить неправильный класс? Объясните через работу классификатора.


## 3. Average feature maps на разных глубинах сети

Для одного изображения необходимо визуализировать feature maps (так же это называют активациями, промежуточными представлениями) после первого слоя, слоя в середине сети и последнего (см. `model.features`)

Такие промежуточные представления как правило имеют много каналов, поэтому усредните каждый по **всем каналам** для выбранного изображения. Посмотрите, что из себя представляют и как меняются feature maps, взятые на разных глубинах сети.

In [ ]:
ACTIVATION_IMAGE_PATH = # YOUR CODE HERE
ACTIVATION_LAYERS = {
    "Первый блок": model.features[0],
    "Средний блок": model.features[len(model.features) // 2],
    "Последний блок": model.features[-1],
}

# YOUR CODE HERE

### Вопросы
1. Как отличаются активации в зависимости от слоя? Какие признаки изображения выделяются на первом, промежуточном и последнем слоях?
2. Что теряется при усреднении каналов?
3. Что делать, если усредненная feature map почти однородная? Значит ли это, что слой не извлек полезных признаков?

## 4. Интерпретация работы сети с помощью Grad-CAM
Установите библиотеку [Grad_CAM](https://github.com/jacobgil/pytorch-grad-cam): `pip install grad-cam`. Используйте `pytorch_grad_cam.GradCAM` с последним блоком модели в качестве target layer.

**Правильные примеры:** среди 5 классов из задания 1 выберите по одному правильному примеру каждого класса.

**Ошибочные примеры:** используйте пять заданных файлов ниже.

Постройте две группы графиков: «Правильные» и «Ошибки», в каждой — оригиналы изображений и наложенная heatmap от Grad-CAM. Подпишите истинный и предсказанный классы.
Целевой класс Grad-CAM — **предсказанный класс из CSV**.

Ваша задача — проанализировать работу Grad-CAM и объяснить, насколько корректно тепловая карта локализует объекты и обосновывает предсказанный класс.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

CAM_TARGET_LAYER = model.features[-1]
CAM_CLASS_COUNT = 5
CAM_ERROR_FILES = [
    "images/09965_label_0282.jpg",
    "images/03014_label_0815.jpg",
    "images/36626_label_0700.jpg",
    "images/06065_label_0060.jpg",
    "images/23357_label_0898.jpg",
]

# YOUR CODE HERE

#### Вопросы

1. Чем Grad-CAM отличается от среднего по каналам feature map?
2. Для ошибочных предсказаний предположите, почему сеть выбрала этот класс. Какие могут быть причины ошибок и как их можно избежать?


## Перед отправкой сохраните ноутбук с именем в следующем формате: **03_Фамилия.ipynb**